In [20]:
import requests
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
import faiss
import chromadb
import json
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [1]:
#Replace with nhs stuff and webscraper later

Base_URL = ("https://www.nhs.uk/health-a-to-z/")
#Scrape nhs website

In [29]:
embedding_model = SentenceTransformer('all-miniLM-L6-v2')

doc_embeddings = embedding_model.encode(
    
    documents,
    normalize_embeddings = True
).astype("float32")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1746.08it/s]


In [30]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)

In [31]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)



Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1657.34it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [32]:
def rag_answer(query, top_k=2, threshold=0.3):
    
    query_embedding = embedding_model.encode(
        
        [query],
        normalize_embeddings=True
        
    ).astype("float32")
    
    distances, indices = index.search(query_embedding, top_k)
    
    if distances[0][0] < threshold:
        
        return None, "Sorry I could not find a relevant document to answer your question."
    
    retrieved_docs = [
        
        documents[i]
        for i in indices[0]
        if i != -1
    ]
    
    context = "\n".join(retrieved_docs)
    
    prompt = f"""
    
    Answer the question using only the information in the context.
    
    Context:
    {context}
    
    Question:
    {query}
    
    Answer:
    """
    
    inputs = tokenizer(
        
        prompt,
        return_tensors="pt",
        truncation=True
        
    )
    
    outputs = model.generate(
        
        **inputs,
        max_new_tokens = 50,
        do_sample=False,
        num_beams=4
    )
    
    response = tokenizer.decode(
        
        outputs[0],
        skip_special_tokens=True
        
    ).strip()
    
    
    return context,response

In [33]:
def run_qa_bot():
    
    print('Welcome to the RAG Q&A Bot! Ask a question or type "exit" to quit.')
    
    while True:
        query = input("User: ")
        
        #Maybe include escape key too?
        if query.lower() == "exit":
            
            print("Goodbye!")
            
            break
        
        if not query.strip():
            print("Please enter a valid question")
            continue
        
        context, answer = rag_answer(query)
        
        print(f"HelpBot: {answer}\n")
        
        
        
run_qa_bot()

Welcome to the RAG Q&A Bot! Ask a question or type "exit" to quit.
Please enter a valid question
HelpBot: Sorry I could not find a relevant document to answer your question.

HelpBot: Sorry I could not find a relevant document to answer your question.

Please enter a valid question
Please enter a valid question
Goodbye!
